# AMEX Enterprise Credit Risk Platform
## Notebook 17 — Comprehensive Reporting: Master Platform Rollup
### Phase 1 · Problem Statement 1: Credit Scoring / PD Prediction

CRISP-DM stage: **Deployment / Rollup**. Notebook 17 of 18. Depends on Notebook 01 only (the config file); every other notebook's output (02 through 16) is read opportunistically, exactly like Notebook 15's documentation scan — so this notebook produces one coherent, single-source-of-truth project-completion report, whatever fraction of the platform has actually been run so far.

**What this notebook builds, all from real, live-scanned `notebook_0N_summary.json` files -- nothing hand-typed that could drift from the truth:**

- A **project completion tracker** -- how many of the 15 core pipeline notebooks (02-16) have actually produced a summary artifact on disk, honestly reported as a percentage, not claimed.
- A **schema-aware per-notebook rollup** -- Notebook 17 knows every prior notebook's summary JSON shape individually (they are not uniform) and extracts each one's real key findings: data volumes, champion model and its measured metrics, SHAP/LIME findings, model risk results, Basel/IFRS9 capital and ECL figures, MLOps latency, API self-test outcome, Docker/monitoring/Power BI status, executive ROI figures, and documentation/architecture coverage.
- A **"Project at a Glance" headline KPI panel** -- the single most decision-relevant number from each pillar, pulled live from whichever pillars have actually run.
- A **platform completion chart** and a **per-pillar status chart**.
- A single consolidated **Comprehensive_Reporting_Report.docx** -- the one document a recruiter or hiring manager could read cover-to-cover to understand the entire 18-notebook platform without opening any other file.

**Deliverables:** `comprehensive_rollup.csv`, `project_at_a_glance.json`, `platform_completion_chart.png`, `pillar_status_chart.png`, `comprehensive_reporting_checklist.csv`, and `Comprehensive_Reporting_Report.docx`.

**Run the single code cell below, once.** Idempotent — every output file is overwritten in place on every re-run.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM NOTEBOOK 01 (EVERYTHING ELSE OPTIONAL)
# =============================================================================
import os
import sys
import json
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebook 01 (Everything Else Optional)")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"{CONFIG_PATH} not found.\nFix: run 01_business_understanding.ipynb first -- "
                             f"this notebook reads its pillar directory map.")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]

_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or DETECTED_LOGICAL_CORES
)

# --- Self-heal: a project_config.json written by an older copy of Notebook 01
#     (before Notebooks 17/18 existed) will not have this pillar key yet.
#     Rather than require re-running Notebook 01 first, derive the standard
#     folder name, create it, and persist it back into project_config.json
#     so every later notebook (including 18) also sees it -- this makes the
#     platform self-consistent regardless of which Notebook 01 vintage
#     originally produced the config. ---
_REQUIRED_PILLARS = {"comprehensive_reporting": "Comprehensive_Reporting"}
_config_healed = False
for _key, _folder_name in _REQUIRED_PILLARS.items():
    if _key not in PILLAR_DIRS:
        PILLAR_DIRS[_key] = PROJECT_ROOT / _folder_name
        PROJECT_CONFIG["pillar_dirs"][_key] = str(PILLAR_DIRS[_key])
        _config_healed = True
        print(f"NOTE: '{_key}' was missing from project_config.json (an older Notebook 01 run) -- "
              f"added it automatically as {PILLAR_DIRS[_key]}")
if _config_healed:
    with open(CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(PROJECT_CONFIG, f, indent=2)
    print(f"✅ project_config.json updated in place -- no need to re-run Notebook 01.")

COMPREHENSIVE_DIR = PILLAR_DIRS["comprehensive_reporting"]
COMPREHENSIVE_DIR.mkdir(parents=True, exist_ok=True)

# --- Discover every notebook_0N_summary.json this run can find -- a fully
#     generic scan (same pattern Notebook 15 uses), not a hand-typed list,
#     so this rollup stays honest about exactly how much of the platform
#     has actually been run, and never silently assumes a notebook ran. ---
NOTEBOOK_SUMMARIES = {}
for _p in sorted(ARTIFACTS_DIR.glob("notebook_*_summary.json")):
    try:
        _num = int(_p.stem.split("_")[1])
    except (IndexError, ValueError):
        continue
    with open(_p, "r", encoding="utf-8") as f:
        NOTEBOOK_SUMMARIES[_num] = json.load(f)

# The core pipeline this rollup covers is notebooks 02-16 (15 notebooks).
# Notebook 01 is the prerequisite config (no comparable summary schema),
# Notebook 17 is this notebook itself (its own summary does not exist yet
# at this point in its own run), and Notebook 18 (packaging) has not run.
CORE_NOTEBOOK_NUMBERS = list(range(2, 17))
_core_found = [n for n in CORE_NOTEBOOK_NUMBERS if n in NOTEBOOK_SUMMARIES]
print(f"Notebook summaries found: {sorted(NOTEBOOK_SUMMARIES.keys())}")
print(f"Core pipeline coverage (Notebooks 02-16): {len(_core_found)} / {len(CORE_NOTEBOOK_NUMBERS)} found")
print(f"Comprehensive reporting will be written under: {COMPREHENSIVE_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION, LIBRARY IMPORTS & ADAPTIVE RAM CEILING
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration, Library Imports & Adaptive RAM Ceiling")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")
try:
    from docx import Document
    from docx.shared import Inches
except ImportError:
    missing.append("python-docx")

if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )

print("(Reporting only -- this notebook is I/O-bound rollup generation, not thread-parallelized work.)")


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


_process_start_rss_gb = _rss_gb()
_live_vm = psutil.virtual_memory()
ADAPTIVE_RAM_FRACTION = _resource_limits.get("ram_fraction_cap", 0.90)
MAX_RAM_BYTES = int(_live_vm.available * ADAPTIVE_RAM_FRACTION)
print(f"Adaptive RAM ceiling (this run) : {MAX_RAM_BYTES / 1e9:.2f} GB")
print(f"\nProcess RSS at Section 2 start: {_process_start_rss_gb:.2f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: PLATFORM ARCHITECTURE MAP & PROJECT COMPLETION TRACKER
# =============================================================================
_section("SECTION 3: Platform Architecture Map & Project Completion Tracker")

# --- Authored reference documentation of the real 18-notebook structure this
#     platform's own build process defines (identical list to Notebook 15's
#     architecture map, kept in sync) -- cross-referenced live against which
#     notebooks have actually produced a summary artifact this run. ---
NOTEBOOK_ARCHITECTURE = [
    {"n": 1, "name": "Business Understanding", "stage": "Business Understanding"},
    {"n": 2, "name": "Data Engineering", "stage": "Data Preparation"},
    {"n": 3, "name": "Data Validation & EDA", "stage": "Data Understanding"},
    {"n": 4, "name": "Feature Engineering", "stage": "Data Preparation"},
    {"n": 5, "name": "Model Development", "stage": "Modeling"},
    {"n": 6, "name": "Explainable AI (SHAP/LIME)", "stage": "Evaluation"},
    {"n": 7, "name": "Model Risk Management", "stage": "Evaluation / Governance"},
    {"n": 8, "name": "Basel III / IFRS9 Mapping", "stage": "Evaluation / Regulatory"},
    {"n": 9, "name": "MLOps", "stage": "Deployment"},
    {"n": 10, "name": "FastAPI Deployment", "stage": "Deployment"},
    {"n": 11, "name": "Docker", "stage": "Deployment"},
    {"n": 12, "name": "Monitoring", "stage": "Deployment / Ongoing Monitoring"},
    {"n": 13, "name": "Power BI Dashboard", "stage": "Deployment / Business Intelligence"},
    {"n": 14, "name": "Executive Reports", "stage": "Deployment / Business Reporting"},
    {"n": 15, "name": "Technical Documentation", "stage": "Deployment / Documentation"},
    {"n": 16, "name": "Production Architecture", "stage": "Deployment / Architecture"},
    {"n": 17, "name": "Comprehensive Reporting", "stage": "Deployment / Rollup"},
    {"n": 18, "name": "Repository Packaging", "stage": "Deployment / Packaging"},
]


def _has_run(n: int) -> bool:
    if n == 1:
        return True  # prerequisite config is required just to reach Section 1
    if n == 17:
        return True  # this notebook is running right now
    return n in NOTEBOOK_SUMMARIES


architecture_df = pd.DataFrame([
    {"n": r["n"], "notebook": r["name"], "crisp_dm_stage": r["stage"], "has_run": _has_run(r["n"])}
    for r in NOTEBOOK_ARCHITECTURE
])
architecture_path = COMPREHENSIVE_DIR / "notebook_architecture_map.csv"
architecture_df.to_csv(architecture_path, index=False)

CORE_TOTAL = len(CORE_NOTEBOOK_NUMBERS)
CORE_FOUND = len(_core_found)
CORE_MISSING = [n for n in CORE_NOTEBOOK_NUMBERS if n not in NOTEBOOK_SUMMARIES]
COMPLETION_PCT = round(100.0 * CORE_FOUND / CORE_TOTAL, 1)
PLATFORM_STATUS = "COMPLETE" if CORE_FOUND == CORE_TOTAL else f"IN PROGRESS ({CORE_FOUND}/{CORE_TOTAL} core notebooks)"

print(architecture_df.to_string(index=False))
print(f"\nCore pipeline (Notebooks 02-16) completion: {COMPLETION_PCT}%  ({CORE_FOUND}/{CORE_TOTAL})")
print(f"Missing core notebooks: {CORE_MISSING if CORE_MISSING else '(none)'}")
print(f"Platform status: {PLATFORM_STATUS}")
print(f"\u2705 Saved -> {architecture_path}")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: SCHEMA-AWARE PER-NOTEBOOK ROLLUP -- REAL KEY FINDINGS FROM EACH SUMMARY
# =============================================================================
_section("SECTION 4: Schema-Aware Per-Notebook Rollup")

# --- Every notebook_0N_summary.json this platform writes has its own real,
#     hand-designed schema (Notebook 02's is a data-volume report; Notebook
#     08's is a Basel/IFRS9 capital report; Notebook 14's is an ROI report;
#     and so on) -- there is no uniform shape to flatten generically without
#     losing meaning. So Notebook 17 defines one small, honest extractor
#     function per notebook number, each reading only the real fields that
#     notebook's own build script actually writes. If a summary is malformed
#     or a field this extractor expects is missing, the extractor fails
#     loudly into a labeled fallback -- never a fabricated placeholder. ---

def _fmt_usd(v):
    return f"${v:,.0f}" if isinstance(v, (int, float)) else v


def _fmt_pct(v, digits=2):
    return f"{v:.{digits}%}" if isinstance(v, (int, float)) else v


def _extract_02(s):
    return {
        "Train CSV": f"{s['train_data_csv']['size_gb']} GB -> {s['train_data_csv']['customers_aggregated']:,} customers "
                     f"({s['train_data_csv']['aggregation_seconds']}s)",
        "Test CSV": f"{s['test_data_csv']['size_gb']} GB -> {s['test_data_csv']['customers_aggregated']:,} customers "
                    f"({s['test_data_csv']['aggregation_seconds']}s)",
        "Live default rate (join validation)": _fmt_pct(s["join_validation"]["live_default_rate"]),
        "Internal train/test split": f"{s['internal_split']['ratio']} stratified by target, seed {s['internal_split']['random_seed']}",
    }


def _extract_03(s):
    return {
        "Validation status": s["validation_status"],
        "Checks passed": f"{s['checks_passed']} / {s['checks_total']}",
        "Charts produced": len(s.get("charts", {})),
        "Reports produced": len(s.get("reports", {})),
    }


def _extract_04(s):
    _top_feat = next(iter(s.get("top_5_new_features_by_corr", {})), None)
    return {
        "Trend features (numeric cols covered)": s["trend_features"]["numeric_columns_covered"],
        "Ratio features created": s["ratio_features_count"],
        "Interaction features created": s["interaction_features_count"],
        "Final engineered train shape": s["final_shape"]["train_full_engineered"],
        "Top new feature by correlation": _top_feat,
    }


def _extract_05(s):
    return {
        "Models trained": ", ".join(s.get("models_trained", [])),
        "Models skipped (not installed)": ", ".join(s.get("models_skipped_not_installed", [])) or "(none)",
        "Champion model": s["champion_model"],
        "Champion holdout AUC (measured)": round(s["champion_metrics"].get("holdout_auc", 0), 4),
        "Champion holdout AMEX metric (measured)": round(s["champion_metrics"].get("holdout_amex_metric", 0), 4),
        "CV folds": s["n_cv_folds"],
    }


def _extract_06(s):
    _top5 = s.get("shap_top10_features") or []
    return {
        "Champion model explained": s["champion_model_explained"],
        "SHAP installed / explainer used": f"{s['shap_installed']} / {s['shap_explainer_used']}",
        "LIME installed": s["lime_installed"],
        "Top SHAP features": ", ".join(_top5[:5]) if _top5 else "n/a (shap not installed)",
        "SHAP vs Notebook 05 importance rank correlation (Spearman)": s.get("shap_vs_nb05_spearman_correlation"),
    }


def _extract_07(s):
    return {
        "Champion validated": s["champion_model_validated"],
        "PSI significant-shift features (measured)": s["psi_significant_shift_features"],
        "Rank-ordering inversions (measured)": s["rank_ordering_inversions"],
        "Assigned model risk tier": s["risk_tier"],
        "GPU used for sensitivity analysis": s["gpu_used_for_sensitivity_analysis"],
    }


def _extract_08(s):
    return {
        "Champion (PD source)": s["champion_model"],
        "Measured portfolio avg PD": _fmt_pct(s["measured_portfolio_avg_pd"]),
        "Total RWA (computed)": _fmt_usd(s["total_rwa_usd"]),
        "Total required regulatory capital (computed)": _fmt_usd(s["total_required_capital_usd"]),
        "Total IFRS9 ECL (computed)": _fmt_usd(s["total_ecl_usd"]),
        "IFRS9 stage counts": s.get("ifrs9_stage_counts"),
        "LGD assumption (editable)": _fmt_pct(s.get("lgd_assumption")) if isinstance(s.get("lgd_assumption"), float) else s.get("lgd_assumption"),
    }


def _extract_09(s):
    _lat = s.get("latency_summary", {})
    _sha = s.get("champion_model_sha256")
    return {
        "Champion registered": f"{s['champion_model']} v{s['champion_model_version']}",
        "Registered model SHA256 (first 12 chars)": _sha[:12] if _sha else None,
        "Inference p99 latency (measured)": f"{_lat.get('single_row_p99_ms', 0):.2f} ms" if "single_row_p99_ms" in _lat else None,
        "Batch throughput (measured)": f"{_lat.get('batch_throughput_rows_per_sec', 0):,.0f} rows/sec" if "batch_throughput_rows_per_sec" in _lat else None,
    }


def _extract_10(s):
    _lat = s.get("api_latency_summary", {})
    return {
        "Champion served": s["champion_model"],
        "Live API self-test": "PASSED" if s["api_self_test_passed"] else "FAILED",
        "API p99 latency (measured)": f"{_lat.get('p99_ms', 0):.2f} ms" if "p99_ms" in _lat else None,
    }


def _extract_11(s):
    _probe = s.get("docker_probe_summary", {})
    return {
        "Docker CLI found this run": _probe.get("docker_cli_found"),
        "Docker daemon reachable this run": _probe.get("daemon_reachable"),
        "Real docker build ran": _probe.get("build_ran"),
        "Measured image size": f"{_probe.get('image_size_mb')} MB" if _probe.get("image_size_mb") else "n/a (no daemon reachable)",
        "Dockerfile lint": "ALL PASSED" if s["lint_all_passed"] else "see checklist",
    }


def _extract_12(s):
    return {
        "Champion model": s["champion_model"],
        "Simulated monitoring windows": s["n_monitoring_windows"],
        "Alerts / Watch this run": f"{s['n_alerts']} / {s['n_watch']}",
        "Windows with an ALERT": s["windows_with_alerts"] or "(none)",
        "PSI significant-shift threshold (assumption)": s.get("monitoring_thresholds", {}).get("psi_significant_shift"),
    }


def _extract_13(s):
    return {
        "Champion model": s["champion_model"],
        "Fact table rows (real, from holdout)": f"{s['fact_table_rows']:,}",
        "Risk tier bands (ASSUMPTION)": s.get("risk_tier_bands"),
        "Model version dimension source": s["model_version_source"],
        "Monitoring window dimension source": s["monitoring_window_source"],
    }


def _extract_14(s):
    return {
        "Champion (measured)": s["champion_model"],
        "Measured holdout AMEX metric": round(s["measured_holdout_amex_metric"], 4),
        "Measured holdout top-4% capture": round(s["measured_holdout_top4pct_capture"], 4),
        "Live portfolio default rate (measured)": _fmt_pct(s["live_portfolio_default_rate"]),
        "Total one-time investment (ASSUMPTION)": _fmt_usd(s["total_one_time_investment_usd"]),
        "Monthly recurring cost (ASSUMPTION)": _fmt_usd(s["monthly_recurring_cost_usd"]),
        "Base-scenario 5yr ROI": f"{s['base_5yr_roi_pct']:.0f}%",
        "Base-scenario 5yr net benefit": _fmt_usd(s["base_5yr_cumulative_net_benefit_usd"]),
    }


def _extract_15(s):
    return {
        "Notebooks with a summary found (at Notebook 15's own run time)": s.get("notebooks_with_summary_found"),
        "Data dictionary features documented": s["data_dictionary_features"],
        "API endpoints documented": s["api_endpoints_documented"],
        "Model versions documented": s["model_versions_documented"],
        "Total platform files (live scan)": f"{s['total_platform_files']:,}",
    }


def _extract_16(s):
    return {
        "Technology stack layers built": f"{s['layers_built']} / {s['layers_total']}",
        "Required replicas (real capacity sizing)": s["required_replicas"] if s.get("required_replicas") else "N/A (Notebook 09 has not run)",
        "SLA met by measured latency": s.get("sla_met_by_measured_latency"),
        "Estimated monthly infra cost": _fmt_usd(s["monthly_infra_cost_usd"]),
    }


EXTRACTORS = {
    2: _extract_02, 3: _extract_03, 4: _extract_04, 5: _extract_05, 6: _extract_06,
    7: _extract_07, 8: _extract_08, 9: _extract_09, 10: _extract_10, 11: _extract_11,
    12: _extract_12, 13: _extract_13, 14: _extract_14, 15: _extract_15, 16: _extract_16,
}


def _safe_extract(n, s):
    try:
        return EXTRACTORS[n](s), None
    except (KeyError, TypeError, IndexError) as exc:
        return {"raw_top_level_keys": ", ".join(sorted(s.keys()))}, str(exc)


ROLLUP_ROWS = []
_extraction_errors = {}
for _r in NOTEBOOK_ARCHITECTURE:
    _n = _r["n"]
    if _n not in EXTRACTORS:
        continue  # notebooks 01, 17, 18 have no comparable rollup schema
    if _n in NOTEBOOK_SUMMARIES:
        _highlights, _err = _safe_extract(_n, NOTEBOOK_SUMMARIES[_n])
        if _err:
            _extraction_errors[_n] = _err
        for _k, _v in _highlights.items():
            ROLLUP_ROWS.append({"n": _n, "notebook": _r["name"], "stage": _r["stage"], "metric": _k, "value": str(_v)})
    else:
        ROLLUP_ROWS.append({"n": _n, "notebook": _r["name"], "stage": _r["stage"],
                             "metric": "(status)", "value": "NOT YET RUN -- no summary artifact found"})

rollup_df = pd.DataFrame(ROLLUP_ROWS)
rollup_path = COMPREHENSIVE_DIR / "comprehensive_rollup.csv"
rollup_df.to_csv(rollup_path, index=False)
print(f"Built {len(rollup_df)} rollup rows across {len(EXTRACTORS)} schema-aware extractors.")
if _extraction_errors:
    print(f"Extraction fell back to raw keys for notebook(s): {_extraction_errors}")
print(f"\u2705 Saved -> {rollup_path}")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: "PROJECT AT A GLANCE" -- HEADLINE KPI PANEL
# =============================================================================
_section("SECTION 5: Project At A Glance -- Headline KPI Panel")

# --- The single most decision-relevant number from each pillar this run has
#     actually produced -- every value below is read live from that pillar's
#     own summary artifact; a pillar that has not run simply contributes
#     nothing here (no fabricated placeholder value). ---
AT_A_GLANCE = {}

if 5 in NOTEBOOK_SUMMARIES:
    _s5 = NOTEBOOK_SUMMARIES[5]
    AT_A_GLANCE["champion_model"] = _s5["champion_model"]
    AT_A_GLANCE["champion_holdout_amex_metric"] = round(_s5["champion_metrics"].get("holdout_amex_metric", 0), 4)
    AT_A_GLANCE["champion_holdout_auc"] = round(_s5["champion_metrics"].get("holdout_auc", 0), 4)

if 7 in NOTEBOOK_SUMMARIES:
    AT_A_GLANCE["model_risk_tier"] = NOTEBOOK_SUMMARIES[7]["risk_tier"]

if 8 in NOTEBOOK_SUMMARIES:
    _s8 = NOTEBOOK_SUMMARIES[8]
    AT_A_GLANCE["total_rwa_usd"] = _s8["total_rwa_usd"]
    AT_A_GLANCE["total_required_capital_usd"] = _s8["total_required_capital_usd"]
    AT_A_GLANCE["total_ifrs9_ecl_usd"] = _s8["total_ecl_usd"]

if 9 in NOTEBOOK_SUMMARIES:
    _lat9 = NOTEBOOK_SUMMARIES[9].get("latency_summary", {})
    if "single_row_p99_ms" in _lat9:
        AT_A_GLANCE["inference_p99_latency_ms"] = round(_lat9["single_row_p99_ms"], 2)
    if "batch_throughput_rows_per_sec" in _lat9:
        AT_A_GLANCE["batch_throughput_rows_per_sec"] = round(_lat9["batch_throughput_rows_per_sec"], 0)

if 10 in NOTEBOOK_SUMMARIES:
    AT_A_GLANCE["live_api_self_test_passed"] = NOTEBOOK_SUMMARIES[10]["api_self_test_passed"]

if 11 in NOTEBOOK_SUMMARIES:
    AT_A_GLANCE["dockerfile_lint_all_passed"] = NOTEBOOK_SUMMARIES[11]["lint_all_passed"]

if 12 in NOTEBOOK_SUMMARIES:
    _s12 = NOTEBOOK_SUMMARIES[12]
    AT_A_GLANCE["monitoring_alerts_this_run"] = _s12["n_alerts"]
    AT_A_GLANCE["monitoring_watch_this_run"] = _s12["n_watch"]

if 14 in NOTEBOOK_SUMMARIES:
    _s14 = NOTEBOOK_SUMMARIES[14]
    AT_A_GLANCE["base_5yr_roi_pct"] = _s14["base_5yr_roi_pct"]
    AT_A_GLANCE["base_5yr_net_benefit_usd"] = _s14["base_5yr_cumulative_net_benefit_usd"]

if 16 in NOTEBOOK_SUMMARIES:
    _s16 = NOTEBOOK_SUMMARIES[16]
    AT_A_GLANCE["required_replicas"] = _s16["required_replicas"]
    AT_A_GLANCE["monthly_infra_cost_usd"] = _s16["monthly_infra_cost_usd"]

AT_A_GLANCE["core_pipeline_completion_pct"] = COMPLETION_PCT
AT_A_GLANCE["platform_status"] = PLATFORM_STATUS

at_a_glance_path = COMPREHENSIVE_DIR / "project_at_a_glance.json"
with open(at_a_glance_path, "w", encoding="utf-8") as f:
    json.dump(AT_A_GLANCE, f, indent=2, default=str)
for _k, _v in AT_A_GLANCE.items():
    print(f"  {_k:45s}: {_v}")
print(f"\n\u2705 Saved -> {at_a_glance_path}")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: PLATFORM COMPLETION & PER-PILLAR STATUS CHARTS
# =============================================================================
_section("SECTION 6: Platform Completion & Per-Pillar Status Charts")

PROBLEM_NAME = "Phase 1 \u00b7 Problem 1 -- Credit Scoring / PD Prediction"
VIZ = {"surface": "#fcfcfb", "text_primary": "#0b0b0b", "text_secondary": "#52514e", "grid": "#e3e2dd",
       "cat_blue": "#2a78d6", "cat_green": "#3a9e5f", "cat_grey": "#9c9b96", "cat_red": "#c0392b"}


def _style_axes(ax):
    ax.set_facecolor(VIZ["surface"])
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.tick_params(colors=VIZ["text_secondary"], labelsize=8)
    ax.grid(axis="y", color=VIZ["grid"], linewidth=0.6, alpha=0.7)


# --- Chart 1: donut of core-pipeline completion (Notebooks 02-16) ---
fig1, ax1 = plt.subplots(figsize=(6, 6), dpi=150)
fig1.set_facecolor(VIZ["surface"])
_sizes = [CORE_FOUND, CORE_TOTAL - CORE_FOUND]
_colors = [VIZ["cat_green"], VIZ["cat_grey"]]
_labels = [f"Complete ({CORE_FOUND})", f"Remaining ({CORE_TOTAL - CORE_FOUND})"]
_is_single_wedge = (CORE_TOTAL - CORE_FOUND == 0) or (CORE_FOUND == 0)
if CORE_TOTAL - CORE_FOUND == 0:
    _sizes, _colors, _labels = [CORE_FOUND], [VIZ["cat_green"]], [f"Complete ({CORE_FOUND})"]
elif CORE_FOUND == 0:
    _sizes, _colors, _labels = [CORE_TOTAL], [VIZ["cat_grey"]], [f"Remaining ({CORE_TOTAL})"]
# A single full wedge makes autopct's own "100%" label redundant with (and
# visually overlapping) the big center label below -- only show autopct
# when the donut genuinely has more than one slice to distinguish.
_autopct_fmt = None if _is_single_wedge else "%1.0f%%"
# ax.pie() returns a 2-tuple (wedges, texts) when autopct is None, and a
# 3-tuple (wedges, texts, autotexts) when it is set -- unpack accordingly.
_pie_result = ax1.pie(
    _sizes, colors=_colors, autopct=_autopct_fmt, startangle=90,
    wedgeprops={"width": 0.42, "edgecolor": VIZ["surface"], "linewidth": 3},
    textprops={"color": "white", "fontsize": 10, "weight": "bold"},
)
ax1.text(0, 0.05, f"{COMPLETION_PCT:.0f}%", ha="center", va="center", fontsize=26, weight="bold", color=VIZ["text_primary"])
ax1.text(0, -0.15, "core pipeline", ha="center", va="center", fontsize=10, color=VIZ["text_secondary"])
ax1.legend(_labels, loc="lower center", bbox_to_anchor=(0.5, -0.12), fontsize=8, frameon=False, ncol=2)
ax1.set_title(f"{PROBLEM_NAME}\nCore Pipeline Completion (Notebooks 02-16)", fontsize=10, color=VIZ["text_primary"])
fig1.tight_layout()
completion_chart_path = COMPREHENSIVE_DIR / "platform_completion_chart.png"
fig1.savefig(completion_chart_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig1)

# --- Chart 2: per-pillar horizontal status bars, all 18 notebooks ---
fig2, ax2 = plt.subplots(figsize=(9, 7), dpi=150)
_style_axes(ax2)
_y_pos = list(range(len(NOTEBOOK_ARCHITECTURE)))[::-1]
_bar_colors = [VIZ["cat_green"] if r["has_run"] else VIZ["cat_grey"]
               for r in ([{"has_run": _has_run(r["n"])} for r in NOTEBOOK_ARCHITECTURE])]
_bar_labels = [f"NB{r['n']:02d} -- {r['name']}" for r in NOTEBOOK_ARCHITECTURE]
ax2.barh(_y_pos, [1] * len(NOTEBOOK_ARCHITECTURE), color=_bar_colors, height=0.65)
ax2.set_yticks(_y_pos)
ax2.set_yticklabels(_bar_labels, fontsize=8, color=VIZ["text_primary"])
ax2.set_xlim(0, 1); ax2.set_xticks([])
ax2.set_title(f"{PROBLEM_NAME}\nPer-Notebook Status (Live, All 18 Notebooks)", fontsize=10, color=VIZ["text_primary"])
_legend_patches = [plt.Rectangle((0, 0), 1, 1, facecolor=VIZ["cat_green"], label="Has run"),
                    plt.Rectangle((0, 0), 1, 1, facecolor=VIZ["cat_grey"], label="Not yet run")]
ax2.legend(handles=_legend_patches, loc="lower right", fontsize=8, frameon=False)
fig2.tight_layout()
pillar_chart_path = COMPREHENSIVE_DIR / "pillar_status_chart.png"
fig2.savefig(pillar_chart_path, dpi=150, facecolor=VIZ["surface"])
plt.show(); plt.close(fig2)

print(f"\u2705 Saved -> {completion_chart_path}")
print(f"\u2705 Saved -> {pillar_chart_path}")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: COMPREHENSIVE REPORTING READINESS CHECKLIST
# =============================================================================
_section("SECTION 7: Comprehensive Reporting Readiness Checklist")

comprehensive_checklist = [
    {"dimension": "Notebook Architecture Map Built", "status": "Pass", "evidence": f"{len(architecture_df)} notebooks mapped"},
    {"dimension": "Core Pipeline Completion Tracked (Live)", "status": "Pass",
     "evidence": f"{CORE_FOUND}/{CORE_TOTAL} core notebooks found ({COMPLETION_PCT}%)"},
    {"dimension": "Schema-Aware Per-Notebook Rollup", "status": "Pass" if len(rollup_df) > 0 else "Fallback (no notebooks have run yet)",
     "evidence": f"{len(rollup_df)} rollup rows across {len(EXTRACTORS)} extractors"},
    {"dimension": "Extraction Ran Without Silent Failures", "status": "Pass" if not _extraction_errors else "Fallback (see raw keys)",
     "evidence": f"{len(_extraction_errors)} notebook(s) fell back" if _extraction_errors else "0 fallbacks"},
    {"dimension": "Project-At-A-Glance KPI Panel", "status": "Pass", "evidence": f"{len(AT_A_GLANCE)} KPIs populated"},
    {"dimension": "Completion & Pillar Status Charts", "status": "Pass", "evidence": "2 charts generated"},
]
comprehensive_checklist_df = pd.DataFrame(comprehensive_checklist)
comprehensive_checklist_path = COMPREHENSIVE_DIR / "comprehensive_reporting_checklist.csv"
comprehensive_checklist_df.to_csv(comprehensive_checklist_path, index=False)
print(comprehensive_checklist_df.to_string(index=False))
print(f"\u2705 Saved -> {comprehensive_checklist_path}")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: WORD REPORT -- COMPREHENSIVE_REPORTING_REPORT.DOCX
# =============================================================================
_section("SECTION 8: Word Report -- Comprehensive_Reporting_Report.docx")


def _add_heading(doc, text, level=1):
    return doc.add_heading(text, level=level)


def _add_kv_table(doc, data: dict):
    table = doc.add_table(rows=0, cols=2)
    table.style = "Light Grid Accent 1"
    for k, v in data.items():
        row = table.add_row().cells
        row[0].text = str(k).replace("_", " ").title()
        row[1].text = "" if v is None else str(v)
    return table


def _add_table_from_df(doc, df, max_rows=40):
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = "Light Grid Accent 1"
    hdr = table.rows[0].cells
    for i, col in enumerate(df.columns):
        hdr[i].text = str(col).replace("_", " ").title()
    for _, row in df.head(max_rows).iterrows():
        cells_ = table.add_row().cells
        for i, col in enumerate(df.columns):
            cells_[i].text = "" if pd.isna(row[col]) else str(row[col])
    if len(df) > max_rows:
        doc.add_paragraph(f"... and {len(df) - max_rows} more row(s) -- see the full CSV for the complete table.")
    return table


report = Document()
report.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
report.add_paragraph("Comprehensive Reporting -- Master Platform Rollup (Notebook 17)")
report.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
report.add_paragraph(f"Platform status: {PLATFORM_STATUS}")

_add_heading(report, "1. Project At A Glance", level=1)
report.add_paragraph(
    "Every value below is read live from that pillar's own summary artifact this run -- a pillar "
    "that has not yet been run simply contributes nothing here, rather than a fabricated placeholder."
)
_add_kv_table(report, AT_A_GLANCE)

_add_heading(report, "2. Core Pipeline Completion", level=1)
report.add_picture(str(completion_chart_path), width=Inches(4.5))
report.add_paragraph(f"{CORE_FOUND} of {CORE_TOTAL} core pipeline notebooks (02-16) have produced a real summary artifact.")
if CORE_MISSING:
    report.add_paragraph(f"Not yet run: {', '.join('NB' + str(n).zfill(2) for n in CORE_MISSING)}")

_add_heading(report, "3. Per-Notebook Status (All 18 Notebooks)", level=1)
report.add_picture(str(pillar_chart_path), width=Inches(6.3))
_add_table_from_df(report, architecture_df)

_add_heading(report, "4. Per-Notebook Detailed Rollup", level=1)
report.add_paragraph(
    "Real key findings extracted from each notebook's own summary artifact via a schema-aware "
    "extractor written specifically for that notebook's real output shape."
)
for _r in NOTEBOOK_ARCHITECTURE:
    _n = _r["n"]
    if _n not in EXTRACTORS:
        continue
    _add_heading(report, f"4.{_n - 1}. Notebook {_n:02d} -- {_r['name']}", level=2)
    if _n in NOTEBOOK_SUMMARIES:
        _sub_df = rollup_df[rollup_df["n"] == _n][["metric", "value"]]
        _add_table_from_df(report, _sub_df, max_rows=20)
    else:
        report.add_paragraph("NOT YET RUN -- no summary artifact found for this notebook.")

_add_heading(report, "5. Comprehensive Reporting Readiness Checklist", level=1)
_add_table_from_df(report, comprehensive_checklist_df)

report_path = COMPREHENSIVE_DIR / "Comprehensive_Reporting_Report.docx"
report.save(str(report_path))
print(f"\u2705 Saved -> {report_path}")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: VERIFICATION -- INTEGRITY CHECKS ON EVERYTHING THIS NOTEBOOK WROTE
# =============================================================================
_section("SECTION 9: Verification")

_checks_passed = True


def _check(label, condition, detail=""):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        _checks_passed = False
        print(f"\u274c {label}  {detail}")


_check("Architecture map covers all 18 notebooks", len(architecture_df) == 18, f"({len(architecture_df)})")
_check("Core completion percentage is honestly bounded [0, 100]", 0.0 <= COMPLETION_PCT <= 100.0, f"({COMPLETION_PCT})")
_check("Core completion matches the real found/total count", COMPLETION_PCT == round(100.0 * CORE_FOUND / CORE_TOTAL, 1))
_check("Rollup row count matches non-empty extractors", len(rollup_df) > 0 or CORE_FOUND == 0)
_check("Every found core notebook appears in the rollup", set(rollup_df.loc[rollup_df['metric'] != '(status)', 'n'].unique()) <= set(_core_found))
_check("At-a-glance KPI panel is present (always carries completion_pct + platform_status, plus any real pillar KPIs)",
       len(AT_A_GLANCE) >= 2)

_expected_files = [architecture_path, rollup_path, at_a_glance_path, completion_chart_path,
                    pillar_chart_path, comprehensive_checklist_path, report_path]
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)

if not _checks_passed:
    raise RuntimeError("One or more Notebook 17 verification checks failed. See \u274c lines above.")

print("\nAll Notebook 17 checks passed.")
print("\n\u2705 Section 9 complete.")


# =============================================================================
# SECTION 10: RESOURCE / PERFORMANCE REPORT
# =============================================================================
_section("SECTION 10: Resource / Performance Report")

_final_rss_gb = _rss_gb()
performance_report = {
    "warp_thread_count_configured": WARP_THREAD_COUNT,
    "adaptive_ram_ceiling_gb": round(MAX_RAM_BYTES / 1e9, 2),
    "process_rss_at_start_gb": round(_process_start_rss_gb, 2),
    "process_rss_at_end_gb": round(_final_rss_gb, 2),
    "core_notebooks_found": sorted(_core_found),
}
performance_report_path = ARTIFACTS_DIR / "notebook_17_performance_report.json"
with open(performance_report_path, "w", encoding="utf-8") as f:
    json.dump(performance_report, f, indent=2)
print(f"Process RSS: {_process_start_rss_gb:.2f} GB (start) -> {_final_rss_gb:.2f} GB (end)")
print(f"\u2705 Saved -> {performance_report_path}")
print("\n\u2705 Section 10 complete.")


# =============================================================================
# SECTION 11: WRITE NOTEBOOK 17 SUMMARY ARTIFACT (for Notebook 18's packaging)
# =============================================================================
_section("SECTION 11: Write Notebook 17 Summary Artifact")

notebook_17_summary = {
    "notebook": "17_comprehensive_reporting", "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "core_pipeline_completion_pct": COMPLETION_PCT, "core_notebooks_found": sorted(_core_found),
    "core_notebooks_missing": CORE_MISSING, "platform_status": PLATFORM_STATUS,
    "at_a_glance": AT_A_GLANCE,
    "output_files": {p.name: str(p) for p in _expected_files + [performance_report_path]},
}
nb17_summary_path = ARTIFACTS_DIR / "notebook_17_summary.json"
with open(nb17_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_17_summary, f, indent=2, default=str)
print(f"\u2705 Saved -> {nb17_summary_path}")
print("\n\u2705 Section 11 complete.")


# =============================================================================
# SECTION 12: COMPLETION SUMMARY
# =============================================================================
_section("SECTION 12: Notebook 17 Complete -- Handoff to Notebook 18")

print("NOTEBOOK 17: COMPREHENSIVE REPORTING -- COMPLETE")
print(f"  Core pipeline completion (02-16) : {COMPLETION_PCT}%  ({CORE_FOUND}/{CORE_TOTAL})")
print(f"  Platform status                  : {PLATFORM_STATUS}")
print(f"  Missing core notebooks           : {CORE_MISSING if CORE_MISSING else '(none)'}")
print(f"  Rollup rows written              : {len(rollup_df)}")
print(f"  At-a-glance KPIs populated       : {len(AT_A_GLANCE)}")
print(f"  Files produced                   : {len(_expected_files) + 2}")
for _p in _expected_files + [performance_report_path, nb17_summary_path]:
    print(f"    - {_p.name}")
print(f"  Next notebook                    : 18_repository_packaging.ipynb")
print("\n\u2705 Ready to proceed.")
